# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Oct 6, 2026 - Day 13: Bayes factors and model comparison

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Today's plan

* Recap: what the Sep 29 review test showed, now that we have the truth
* MCMC again: why Metropolis accepts worse steps, and how to read a chain
* this week: Lab 05, the midterm, one syllabus change
* today's question: when is an extra parameter worth it? ($\sigma_{\rm int}$ in the review test, a jitter in Lab 05, a 4th planet around 47 UMa)
* the denominator in Bayes' rule, and choosing between models with it: Bayes factors and Occam's razor
* computing it: grids, the Laplace approximation, nested sampling
* 47 UMa: three planets or four? Then cosmology's $n_s$, and an aside
* lab defenses, in the room, at the end

Moved to Thursday: detailed balance, $\hat R$ across several chains, and HMC (promised on Day 12)

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats
plt.rcParams.update({'font.size': 13, 'legend.fontsize': 12})
from ipywidgets import interactive_output, FloatLogSlider, IntSlider, HBox, VBox, Layout
from IPython.display import display

def controls_right(fn, **sliders):
    # the plot on the left, its sliders stacked on the right (the Day 12 layout)
    for w in sliders.values():
        w.layout = Layout(width='230px'); w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, sliders), VBox(list(sliders.values()), layout=Layout(justify_content='center', width='240px'))]))

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

# The Sep 29 review-test sample: 45 simulated dwarf AGN (log M*, log M_BH, per-point errors in both),
# lecture/06/data/inclass_day10_dwarf_agn.csv. Truth: alpha = 1.25, beta = 6.40, sigma_int = 0.45 dex
# (simulation mu = 9.4, sig_pop = 0.55), given out in class today and in ASSIGNMENTS.md.
_f = '../06/data/inclass_day10_dwarf_agn.csv'
with open(_f) as _fh:
    _nhead = sum(1 for _line in _fh if _line.startswith('#'))
_d = np.genfromtxt(_f, delimiter=',', skip_header=_nhead, names=True)
x, sx, y, sy = _d['logMstar'], _d['err_logMstar'], _d['logMBH'], _d['err_logMBH']
N = x.size
TRUTH = dict(alpha=1.25, beta=6.40, sig_int=0.45)

# OLS and WLS by the normal equations, exactly as in the test
A = np.vstack([np.ones(N), x - 9.5]).T
_th = np.linalg.inv(A.T @ A) @ A.T @ y
_s2 = np.sum((y - A @ _th)**2) / (N - 2)
ols = (_th[1], _th[0], np.sqrt(_s2 * np.linalg.inv(A.T @ A)[1, 1]))
_Si = np.diag(1 / sy**2)
_cw = np.linalg.inv(A.T @ _Si @ A); _tw = _cw @ A.T @ _Si @ y
wls = (_tw[1], _tw[0], np.sqrt(_cw[1, 1]))
chi2nu_wls = np.sum(((y - A @ _tw) / sy)**2) / (N - 2)

# the five-parameter generative fit, curvature error bars
def _neg2logL(p):
    a, b, ls, m, lt = p
    si2, t2 = np.exp(2 * ls), np.exp(2 * lt)
    dx, dy = x - m, y - (a * (m - 9.5) + b)
    s11 = t2 + sx**2; s22 = a**2 * t2 + si2 + sy**2; s12 = a * t2
    det = s11 * s22 - s12**2
    return np.sum(np.log(det) + (s22 * dx**2 - 2 * s12 * dx * dy + s11 * dy**2) / det)
_r = minimize(_neg2logL, [ols[0], ols[1], np.log(0.3), x.mean(), np.log(x.std())], method='Nelder-Mead',
              options=dict(xatol=1e-8, fatol=1e-8, maxiter=20000))
_p, _h, _H = _r.x, 1e-4, np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        ei = np.zeros(5); ej = np.zeros(5); ei[i] = _h; ej[j] = _h
        _H[i, j] = (_neg2logL(_p + ei + ej) - _neg2logL(_p + ei - ej) - _neg2logL(_p - ei + ej) + _neg2logL(_p - ei - ej)) / (4 * _h * _h)
gen = (_p[0], _p[1], np.sqrt(2 * np.linalg.inv(_H)[0, 0]))
print(f"N = {N}; OLS {ols[0]:.3f} +/- {ols[2]:.3f}; WLS {wls[0]:.3f} +/- {wls[2]:.3f} (chi2/nu = {chi2nu_wls:.1f}); "
      f"generative {gen[0]:.3f} +/- {gen[2]:.3f}, z = {(gen[0] - TRUTH['alpha']) / gen[2]:+.2f}")

## Recap: the review test, now with the truth

* the truth is $\alpha = 1.25$; everyone who submitted reported $1.04 \pm 0.21$ from the generative fit (Q3)
* $z = -1.0$, about one error bar low - halfway reasonable, and that's kind of what the point of an error bar is
* OLS got 0.89 and WLS 0.85, both low because of attenuation
* Q2 predicted OLS at about 80% of the truth; it got 71%, inside its $\pm 0.18$

In [ ]:
# Visual reminder: the review-test sample, three fits and the true line (a "skip" cell: runs, not a slide)
def plot_recap():
    fig, (ax, az) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw=dict(width_ratios=[1.6, 1]))
    xx = np.linspace(x.min() - 0.3, x.max() + 0.3, 20)
    ax.errorbar(x, y, xerr=sx, yerr=sy, fmt='o', color='0.6', ms=3.5, elinewidth=0.6, capsize=0)
    ax.plot(xx, TRUTH['beta'] + TRUTH['alpha'] * (xx - 9.5), 'k:', lw=2.4, label=f"truth: {TRUTH['alpha']:.2f}")
    zs = {}
    for name, (a, b, s), col, ls, lw in [('OLS', ols, 'C2', '-', 2.0), ('WLS', wls, 'C1', '--', 2.4), ('generative', gen, 'C3', '-.', 2.4)]:
        ax.plot(xx, b + a * (xx - 9.5), color=col, ls=ls, lw=lw, label=f'{name}: {a:.2f} $\\pm$ {s:.2f}')
        zs[name] = (a - TRUTH['alpha']) / s
    ax.set_xlabel(r'$\log_{10} M_*$ (observed)'); ax.set_ylabel(r'$\log_{10} M_{\rm BH}$ (observed)'); ax.legend(frameon=False)
    az.barh(list(zs), list(zs.values()), color=['C2', 'C1', 'C3']); az.axvspan(-2, 2, color='0.92', zorder=0); az.axvline(0, color='k', lw=0.8)
    az.set_xlabel(r'$z$ = (fit $-$ truth) / error bar'); az.set_xlim(-6, 2)
    az.text(-1.9, -0.45, 'within 2$\\sigma$', fontsize=11, color='0.4')
    plt.tight_layout()
    plt.show()

In [ ]:
# the Sep 29 sample: three slopes against the truth (dotted), and each one's z-score
plot_recap()

## Q1: why WLS missed by five sigma

* 10 of you trusted WLS ($\pm 0.08$), 8 trusted OLS ($\pm 0.18$)
* WLS assumes the $\sigma_y$ are the whole scatter - here the residuals are much bigger
* two terms are missing: $\sigma_{\rm int}$, the intrinsic scatter, and $\alpha^2\sigma_x^2$ from the $x$ errors

In [ ]:
# Q1's numbers, printed under the slide (a "skip" cell)
def q1_numbers():
    print(f"WLS: chi^2/nu = {chi2nu_wls:.1f}, so the residuals are sqrt({chi2nu_wls:.1f}) = {np.sqrt(chi2nu_wls):.1f}x the y error bars")
    print(f"WLS error bar x sqrt(chi^2/nu) = {wls[2]:.2f} x {np.sqrt(chi2nu_wls):.1f} = {wls[2] * np.sqrt(chi2nu_wls):.2f};  OLS error bar = {ols[2]:.2f}")

In [ ]:
# reduced chi-square of the WLS fit, and what it does to the error bar
q1_numbers()

* WLS quoted $\pm 0.08$ because it assumed the $\sigma_y$ were *all* of the scatter. $\chi^2/\nu = 5$ says the real scatter is $\sqrt{5} = 2.2\times$ bigger
* so the honest WLS error bar is $0.08 \times 2.2 = 0.18$ - the same as OLS's, which measured the scatter from its own residuals
* with $\pm 0.08$, WLS's 0.85 sits $(1.25 - 0.85)/0.08 = 5\sigma$ from the truth; with the honest $\pm 0.18$, about $2\sigma$
* both slopes are still low - that is attenuation from the $x$ errors, which no error bar fixes (Q2)
* Lab 05's jitter $s$ is the same fix: add $s^2$ to every $\sigma_i^2$ and let the data set it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If $\chi^2/\nu \gg 1$, your error bars are too small by $\sqrt{\chi^2/\nu}$. The reduced chi-square here was actually a warning. Look at your data!**

</div>

## Q4: why $x$ and $y$ move together

<img src="images/review_sep29_pgm.png" alt="Probabilistic graphical model for the generative fit: the population parameters mu and sigma_pop point to a latent true stellar mass x_true inside a plate over galaxies; x_true, alpha, beta and sigma_int point to a latent true black-hole mass y_true; x_true and the known error sigma_x point to the observed x; y_true and sigma_y point to the observed y" style="display:block;margin:0 auto;max-height:300px">

<div style="font-size:0.8em; text-align:center; color:#666;">Drawn for this course with matplotlib, the reference answer to Q4.</div>

* the box is a plate: everything inside it is repeated for each galaxy $i$
* most of you drew the nodes right; nobody said where the off-diagonal term comes from
* both observed numbers have the same parent, $x_{\rm true}$, so they share its scatter:
$$\textrm{Cov}(x_i, y_i) = \alpha\,\sigma_{\rm pop}^2, \qquad \textrm{Var}(x_i) = \sigma_{\rm pop}^2 + \sigma_{x,i}^2$$
* this picture is the forward model - i.e. the likelihood, Bayesian or not

## Metropolis: accepting worse steps

* propose $\theta'$; if $P(\theta' \mid D)$ is higher, go there
* if it is lower - maybe that's bad, or maybe we're just unlucky: accept with probability $R = P(\theta' \mid D) / P(\theta \mid D)$ (always, if $R \ge 1$)
* the stochasticity is the point!
* stochasticity allows us to explore the whole surface but spend more time in interesting spots

In [ ]:
# 1-D Metropolis on a known posterior, marking the accepted worse steps (a "skip" cell)
def plot_metropolis(nstep=20000, step=1.5, seed=3):
    rng = np.random.default_rng(seed)
    logp = lambda t: -0.5 * t**2                     # a unit Gaussian posterior, so we know the answer
    chain, worse = np.empty(nstep), np.zeros(nstep, bool)
    t = 3.0
    for k in range(nstep):
        tp = t + step * rng.normal()
        R = np.exp(logp(tp) - logp(t))
        if rng.uniform() < R:
            worse[k] = R < 1
            t = tp
        chain[k] = t
    fig, (ax, az) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw=dict(width_ratios=[1.6, 1]))
    kk = np.arange(300)
    ax.plot(kk, chain[:300], color='0.4', lw=0.9)
    ax.plot(kk[worse[:300]], chain[:300][worse[:300]], 'o', color='C3', ms=4, label='accepted a worse step')
    ax.set_xlabel('step'); ax.set_ylabel(r'$\theta$'); ax.legend(frameon=False, loc='upper right')
    tt = np.linspace(-4, 4, 200)
    az.hist(chain[500:], 50, density=True, color='0.75', label='samples')
    az.plot(tt, stats.norm.pdf(tt), 'k', lw=1.6, label='true posterior')
    az.set_xlabel(r'$\theta$'); az.legend(frameon=False, loc='upper left')
    plt.tight_layout()
    plt.show()
    moved = np.diff(chain) != 0
    print(f"accepted {moved.mean():.0%} of proposals; {worse.sum() / moved.sum():.0%} of the accepted steps went to a lower posterior")

In [ ]:
# trace with every accepted worse step in red, and the histogram of samples against the true posterior
plot_metropolis()

* without the red steps the chain goes to $\theta = 0$ and stays there
* MCMC is a terrible optimizer. If you just want the "best-fit", some local/global optimizer is often quicker
* Lab 05 is this in seven dimensions, with `emcee`

## Before you trust a chain: Day 12's RX J1131 run

* the same Metropolis chain as Day 12: delay $\Delta t$ and jitter $\ln s$, 10000 steps
* look at the trace and throw away the burn-in, the steps before the chain forgets its start
* check the acceptance - 0.49 here; Day 12's targets were 0.44 in 1-D and 0.23 in many dimensions (0.2 to 0.5 for `emcee`, Foreman-Mackey et al. 2013)

In [ ]:
# Day 12's cached Metropolis chain for RX J1131-1231: trace with the burn-in shaded (a "skip" cell)
import emcee, corner
_z = np.load('../06/data/day12_cache.npz')
chain_lens, acc_lens, BURN = _z['lens/chain'], float(_z['lens/acc']), 300
post_lens = chain_lens[BURN:]
tau_lens = np.array([emcee.autocorr.integrated_time(post_lens[:, i], quiet=True)[0] for i in range(2)])
def plot_trace_lens():
    fig, (ax, az) = plt.subplots(2, 1, figsize=(11, 4.4), sharex=True)
    for a, i, lab in [(ax, 0, r'$\Delta t$ [d]'), (az, 1, r'$\ln s$')]:
        a.plot(chain_lens[:2000, i], lw=0.7, color='C0')
        a.axvspan(0, BURN, color='C3', alpha=0.15, label='burn-in (dropped)' if i == 0 else None)
        a.set_ylabel(lab)
    ax.legend(frameon=False, loc='lower right'); az.set_xlabel('step')
    plt.tight_layout()
    plt.show()
    print(f"acceptance {acc_lens:.2f};  tau = {tau_lens.max():.0f} steps, so {post_lens.shape[0]} samples are worth about {round(post_lens.shape[0] / tau_lens.max(), -2):.0f} independent ones")

In [ ]:
# the first 2000 steps of Day 12's chain; the shaded start is the burn-in we drop
plot_trace_lens()

* measure $\tau$, the autocorrelation time - neighboring samples are not independent, so $N_{\rm eff} \approx N/\tau$, about 1100 here
* Lab 05 asks for the chains and $\tau$, then Day 12's posterior predictive check

**Reading the answer off the samples**

* $94.6 \pm 1.3$ d is the 50th, 16th and 84th percentile of the 9700 delay samples
* the jitter column is never used - ignoring it marginalizes over it

In [ ]:
# corner plot of the post-burn-in chain, with the 16/50/84% lines (a "skip" cell)
def plot_corner_lens():
    fig = corner.corner(post_lens, labels=[r'$\Delta t$ [d]', r'$\ln s$'], quantiles=[0.16, 0.5, 0.84], label_kwargs=dict(fontsize=14))
    fig.set_size_inches(5.2, 5.2)
    plt.show()
    lo, mid, hi = np.percentile(post_lens, [16, 50, 84], axis=0)
    print(f"delay = {mid[0]:.1f} +{hi[0]-mid[0]:.1f} / -{mid[0]-lo[0]:.1f} d;  jitter s = {np.exp(mid[1])*1000:.0f} mmag")

In [ ]:
# the corner plot from Day 5, built from Day 12's samples: diagonal = one parameter, off-diagonal = both
plot_corner_lens()

* in Lab 05 the corner plot is $7\times7$, and you quote the mass function $f(M)$ the same way, one value per sample

## This week

* **Lab 05** (the binary star) reached GitHub late, so it is now due **Fri Oct 9 by Noon**
* **midterm**: posted Thu Oct 8, due Wed Oct 14 by Noon
* syllabus change: no separate midterm or final oral defenses; both are graded like a lab
* **Quiz 3** is tentatively Tue Oct 20, on Bayes and MCMC
* Abha's office hours move to Fridays, 1-2 pm, over Zoom
* every date is in `ASSIGNMENTS.md`

## The denominator in Bayes' rule

$$P(\theta \mid D, M) = \frac{P(D \mid \theta, M)\, P(\theta \mid M)}{P(D \mid M)}, \qquad P(D \mid M) = \int P(D \mid \theta, M)\, P(\theta \mid M)\, d\theta$$

* Metropolis only used the ratio $R$, so it never needed this denominator. In model comparison problems, the proportionality constant must be known
* Day 12 wrote it $P(D)$. With more than one model on the table we write $P(D \mid M)$, where $M$ is everything behind the likelihood and the prior
* the evidence: before you saw the data, how probable did model $M$ say this dataset would be?
* it is the integral over the likelihood for *all possible values of $\theta$*, weighted by the prior - the likelihood averaged over everything the model allows
* this means we could have called it the Fully Marginalized Likelihood (FML) instead of the evidence

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**For model fitting, you can ignore the FML as a simple normalization term. In model selection, the FML can become important.**

</div>

## What $P(D \mid M)$ measures

In [ ]:
# What P(D | M) means: each model spreads one unit of probability over the datasets it could have produced (a "skip" cell)
def plot_evidence_cartoon():
    D = np.linspace(-10, 10, 600)
    simple = stats.norm.pdf(D, 0, 1.2)                    # a model with little freedom: predicts a narrow range of data
    flexible = stats.norm.pdf(D, 0, 5.0)                  # a model with more freedom: can produce almost anything
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(D, simple, 'C0', lw=2.6, label=r'$M_0$, simple: few datasets, each likely')
    ax.plot(D, flexible, 'C1', lw=2.6, ls='--', label=r'$M_1$, flexible: many datasets, each less likely')
    for d0, txt in [(0.8, 'data like these:\n$M_0$ wins'), (5.5, 'data like these:\n$M_1$ wins')]:
        ax.axvline(d0, color='0.3', lw=1.2, ls=':')
        ax.text(d0 + 0.25, 0.42, txt, fontsize=12, va='top', color='0.25')
    ax.set_xlabel('every dataset you could have observed (one axis, schematically)')
    ax.set_ylabel(r'$P(D \mid M)$'); ax.set_yticks([]); ax.set_xticks([]); ax.set_ylim(0, 0.46)
    ax.legend(frameon=False, loc='upper left', fontsize=12, bbox_to_anchor=(0, 0.75))
    plt.tight_layout()
    plt.show()

In [ ]:
# P(D|M) for a simple and a flexible model; both curves have area 1, so freedom costs height
plot_evidence_cartoon()

* each model has exactly one unit of probability to spread over every dataset it could produce
* a flexible model (more parameters, or wider priors) can produce more datasets, so it gives each one less
* if two models explain the data equally well, the simpler one wins - $P(D \mid M)$ applies Occam's razor on its own, with no penalty added by hand
* today's question again: $\sigma_{\rm int}$ in the review test, a jitter in Lab 05, a fourth planet around 47 UMa - is the extra freedom worth it?

## Bayes factors

$$\frac{P(M_1 \mid D)}{P(M_0 \mid D)} = \underbrace{\frac{P(D \mid M_1)}{P(D \mid M_0)}}_{B_{1,0},\ \textrm{Bayes factor}} \times \frac{P(M_1)}{P(M_0)}, \qquad B_{1,0} = \frac{\int P(D \mid \theta_1, M_1)\, P(\theta_1 \mid M_1)\, d\theta_1}{\int P(D \mid \theta_0, M_0)\, P(\theta_0 \mid M_0)\, d\theta_0}$$

* $B_{1,0}$ is "B one-zero": the evidence for model 1 on top, model 0 on the bottom. Each model integrates over its *own* parameters
* convention here: $M_0$ is the simpler model, $M_1$ the one with the extra parameter. $B_{1,0} > 1$ favors $M_1$; $B_{0,1} = 1/B_{1,0}$. Papers differ, so always check which model is on top
* $P(M)$ is how much you believed a model before the data; a ratio of two such probabilities is called odds. The data turn prior odds into posterior odds by multiplying by $B$ (we use $B$; $K$ is Lab 05's RV amplitude)
* we quote $2\ln B_{1,0}$, because with Gaussian errors $-2\ln L = \chi^2 + $ const. That puts it on the same scale as $\Delta\chi^2$ and the $\Delta$BIC of Days 8 and 9: positive favors $M_1$, negative favors $M_0$

In [ ]:
# The Kass & Raftery ladder for 2 ln B, with the odds each rung means (a "skip" cell)
LADDER = [(-10, -6, '$M_0$: strong', '#e2e2e2'), (-6, -2, '$M_0$: positive', '#efefef'), (-2, 2, 'weak', '#ffffff'),
          (2, 6, '$M_1$: positive', '#eef4fb'), (6, 10, '$M_1$: strong', '#dde9f6'), (10, 14, '$M_1$: very strong', '#c6dbef')]
def plot_ladder():
    fig, ax = plt.subplots(figsize=(11, 1.9))
    for lo, hi, lab, col in LADDER:
        ax.axvspan(lo, hi, color=col, ec='0.6', lw=0.6)
        ax.text((lo + hi) / 2, 0.68, lab, ha='center', va='center', fontsize=12)
    for v in (-6, -2, 2, 6, 10):
        B = np.exp(v / 2)
        ax.text(v, 0.22, f'B = {B:.0f}' if B >= 1 else f'B = 1/{1/B:.0f}', ha='center', va='center', fontsize=11, color='C3')
    ax.axvline(0, color='k', lw=1.4)
    ax.set_xlim(-10, 14); ax.set_ylim(0, 1); ax.set_yticks([])
    ax.set_xlabel(r'$2\ln B_{1,0}$    ($> 0$ favors $M_1$, $< 0$ favors $M_0$;  $\approx$ BIC$_0 -$ BIC$_1$, since lower BIC wins)')
    plt.tight_layout()
    plt.show()

In [ ]:
# 2 ln B on Day 8's ladder; red: the Bayes factor at each boundary
plot_ladder()

* above 0 favors $M_1$, below 0 favors $M_0$. $2\ln B = 6$ means $B = 20$
* this is Day 8's ladder for $\Delta$BIC (Kass &amp; Raftery 1995), now with the odds each rung means if you started at even odds. "Positive" is their name for the middle rung (it does not mean $> 0$)
* BIC is "lower wins", so $2\ln B_{1,0} \approx \textrm{BIC}_0 - \textrm{BIC}_1$ for large $N$: Day 9's $\Delta$BIC = 15.8 for the CO line, taken the same way round (why: two slides on)

## Occam's razor

* let's say we have 20 data points with known unit errors, and two models
* $M_0$: the mean is exactly zero. $M_1$: the mean is free, with a Gaussian prior of width $\sigma_{\rm prior}$ centered on zero
* the data and the likelihood are fixed; the slider only changes $\sigma_{\rm prior}$. Try 5, then 50, then 500, then all the way left

In [ ]:
# Evidence for a free mean vs a fixed one, as the Gaussian prior on the mean widens (a "skip" cell)
_rng = np.random.default_rng(33)
occ_d = _rng.normal(0.7, 1.0, 20)                          # 20 points, true mean 0.7, unit errors, known
occ_n, occ_mean = occ_d.size, occ_d.mean()
occ_sig_post = 1 / np.sqrt(occ_n)                          # width of the likelihood for the mean
def occ_two_lnB(sp):
    # M1: mean ~ N(0, sp^2).  P(D|M1)/P(D|M0) depends on the data only through the sample mean:
    # it is N(mean; 0, sp^2 + 1/n) / N(mean; 0, 1/n), the two models' predictions for the sample mean
    sp = np.asarray(sp, float)
    return 2 * (stats.norm.logpdf(occ_mean, 0, np.sqrt(sp**2 + 1 / occ_n)) - stats.norm.logpdf(occ_mean, 0, np.sqrt(1 / occ_n)))
def plot_occam(sigma_prior=5.0):
    fig, (a0, ax, az) = plt.subplots(1, 3, figsize=(13, 3.7), gridspec_kw=dict(width_ratios=[1, 1.1, 1.3]))
    a0.errorbar(np.arange(occ_n), occ_d, yerr=1, fmt='o', color='0.3', ms=4, capsize=0)
    a0.axhline(0, color='k', lw=1.2, ls=':', label='$M_0$: mean = 0')
    a0.axhline(occ_mean, color='C0', lw=2, label=f'$M_1$: mean free, best {occ_mean:.2f}')
    a0.set_xlabel('data point'); a0.set_ylabel('value'); a0.set_ylim(-2.2, 5.2); a0.legend(frameon=False, loc='upper left', fontsize=11)
    mm = np.linspace(-3, 3, 1200)
    prior = stats.norm.pdf(mm, 0, sigma_prior)
    post = prior * stats.norm.pdf(mm, occ_mean, occ_sig_post); post /= np.trapezoid(post, mm)
    ax.plot(mm, post, 'C3', lw=2.6, label='posterior')
    ax.plot(mm, prior, 'C0', ls='--', lw=2, label='prior')
    ax.set_xlabel('mean'); ax.set_ylabel('probability density'); ax.set_xlim(-3, 3); ax.set_xticks([-3, -2, -1, 0, 1, 2, 3])
    ax.set_ylim(0, 1.15 * max(post.max(), prior.max()))
    ax.legend(frameon=False, loc='upper left', fontsize=11)
    ax.set_title(rf'$\sigma_{{\rm prior}}$ = {sigma_prior:.3g}: prior height {prior.max():.2g}', fontsize=12)
    widths = np.logspace(-1, 3, 200)
    for lo, hi, lab, col in LADDER[:5]:
        lo2, hi2 = max(lo, -8), min(hi, 8)
        az.axhspan(lo2, hi2, color=col, zorder=0); az.text(0.115, (lo2 + hi2) / 2 + (1 if lab == 'weak' else 0), lab, fontsize=10, va='center', color='0.35')
    az.axhline(0, color='k', lw=0.8)
    az.semilogx(widths, occ_two_lnB(widths), 'C3', lw=2.2)
    v = float(occ_two_lnB(sigma_prior))
    az.plot(sigma_prior, v, 'o', color='C3', ms=9); az.annotate(f'{v:+.1f}', (sigma_prior, v), xytext=(6, 6), textcoords='offset points', fontsize=13)
    az.set_ylim(-8, 8); az.set_xlabel(r'prior width $\sigma_{\rm prior}$'); az.set_ylabel(r'$2\ln B_{1,0}$, free mean vs zero')
    plt.tight_layout()
    plt.show()

In [ ]:
# left: data and both models; middle: prior (dashed) and posterior on the mean; right: 2 ln B as the prior widens
controls_right(plot_occam, sigma_prior=FloatLogSlider(value=5.0, base=10, min=-1, max=3, step=0.05, continuous_update=False, description='σ_prior'))

* $L_{\max}$ is the best-fit likelihood. The data alone pin the mean to $\sigma_{\rm post} = 1/\sqrt{20} = 0.22$. Past that, the posterior doesn't move, but $2\ln B$ goes from $+3.2$ at $\sigma_{\rm prior} = 5$ to $-5.9$ at 500
* $P(D \mid M_1) \approx L_{\max} \times \sigma_{\rm post}/\sigma_{\rm prior}$ - you've made the likelihood pay, as a penalty term, for putting in a wider prior
* $P(D \mid M_0) = L_{\max}\, e^{-\lambda^2/2}$, where $\lambda = 3.1$ is how many $\sigma$ the best mean is from zero. so $2\ln B_{1,0} \approx \lambda^2 - 2\ln(\sigma_{\rm prior}/\sigma_{\rm post}) = 9.5 - 6.2 = +3.3$
* at $\sigma_{\rm prior} = 500$ a mean $3\sigma$ from zero loses to "exactly zero" - Lindley's paradox: a wide enough prior makes any detection lose
* far left, the prior squeezes $M_1$ onto $M_0$, and $2\ln B$ falls toward 0 (+1.4 at the slider's left end)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The posterior doesn't change with $\sigma_{\rm prior}$. The Bayes factor does!**

</div>

## Computing $P(D \mid M)$

* **on a grid**, as the $\sigma_{\rm prior}$ widget did. Computing this integral can become extremely costly (a manifestation of the *curse of dimensionality*) - 100 points per axis in Lab 05's 7 dimensions is $10^{14}$ likelihood calls
* **not from `emcee`** - the constant of proportionality is not important to Metropolis, so it never computes it
* **the Laplace approximation** - cheap, when the posterior is close to a Gaussian (next)
* **nested sampling** - when it isn't, or has several peaks (after that)

## The Laplace approximation

* treat the posterior as a Gaussian at its peak $\hat\theta$ ($d$ parameters), and integrate the Gaussian exactly
$$P(D \mid M) \approx L_{\max} \times \underbrace{P(\hat\theta \mid M)\,(2\pi)^{d/2}\,|C|^{1/2}}_{\textrm{Occam factor}}, \qquad C = I^{-1}, \quad I_{ij} = -\frac{\partial^2 \ln L}{\partial\theta_i\,\partial\theta_j}\Big|_{\hat\theta}$$
* $C$ is Day 5's covariance matrix: the inverse of the curvature of $\ln L$ at the peak (Day 5's *observed* information; averaged over datasets it is the Fisher information), the same matrix behind your Fisher error bars ($|C|$ is its determinant, $P(\hat\theta \mid M)$ the prior density at the peak)
* so the Laplace approximation turns the Fisher error bars into an evidence: the Occam factor is about (posterior volume) / (prior volume), in 1-D the $\sigma_{\rm post}/\sigma_{\rm prior}$ from the widget
* large $N$, and a prior about as wide as what one data point alone would allow: $-2\ln P(D \mid M) \to -2\ln L_{\max} + d\ln N$, i.e. Day 8's BIC

In [ ]:
# The Laplace approximation on a skewed posterior: 3 X-ray photons, rate lam, flat prior on [0, 20] (a "skip" cell)
def plot_laplace(k=3, lam_max=20.0):
    lam = np.linspace(0.001, 14, 2000)
    like = stats.poisson.pmf(k, lam)                       # L(lam) for k observed photons
    Z_exact = np.trapezoid(like, lam) / lam_max            # integral of L x prior, prior density 1/20
    lam_hat = k                                            # the peak
    sig = np.sqrt(lam_hat**2 / k)                          # 1/sqrt(curvature of -ln L at the peak): Day 5's error bar
    Z_lap = stats.poisson.pmf(k, lam_hat) * np.sqrt(2 * np.pi) * sig / lam_max
    fig, ax = plt.subplots(figsize=(7.4, 3.4))
    ax.plot(lam, like / like.max(), 'k', lw=2.4, label='likelihood (exact)')
    ax.plot(lam, np.exp(-0.5 * ((lam - lam_hat) / sig)**2), 'C3', ls='--', lw=2.2, label=f'Laplace: Gaussian at the peak, $\\sigma$ = {sig:.2f}')
    ax.set_xlabel(r'photon rate $\lambda$'); ax.set_ylabel('likelihood / max'); ax.set_ylim(0, 1.25)
    ax.legend(frameon=False, loc='upper right', fontsize=11)
    ax.set_title(f'$P(D\\mid M)$: exact {Z_exact:.4f}, Laplace {Z_lap:.4f} ({Z_lap / Z_exact - 1:+.0%})', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# 3 photons: the exact likelihood (black) is skewed; the Laplace Gaussian (red dashed) matches its peak and curvature
plot_laplace()

* even for this skewed likelihood (3 photons, a flat prior on the rate from 0 to 20), the Laplace evidence is off by 3%
* it fails where Fisher fails (Day 5's "Where Fisher stops": banana-shaped posteriors), and also for several peaks or a peak pressed against a prior edge

## Nested sampling

* $Z$ is the usual shorthand for $P(D \mid M)$ in these codes
* Skilling (2004): turn the $d$-dimensional integral into a 1-D one, over the prior volume $X$
$$P(D \mid M) = \int_0^1 L(X)\, dX, \qquad X(\lambda) = \int_{L(\theta) > \lambda} P(\theta \mid M)\, d\theta$$
* start with $n_{\rm live}$ "live" points drawn from the prior. Drop the lowest-likelihood one (now a "dead" point), and replace it with a new prior draw at higher likelihood
* each kill shrinks $X$ by about $e^{-1/n_{\rm live}}$. Add up $L \times \Delta X$ as you go, and stop when the live points can't change the sum
* this is also a sampler, but is not actually a Markov Chain method. The dead points, weighted by $L\,\Delta X$, are posterior samples too
* `dynesty` (Speagle 2020), `MultiNest` and `UltraNest` differ in how they find that higher-likelihood draw - the hard part in many dimensions

<img src="images/speagle2020_fig2_top_bottom_crop.png" alt="Two panels from a dynesty nested-sampling run, both against minus the log of the prior volume X: top, a constant number of live points with three insets showing the cloud of live points shrinking onto a narrow ellipse; bottom, the estimated evidence climbing and levelling off at the true value, with shaded uncertainty bands" style="display:block;margin:0 auto;max-height:330px">

<div style="font-size:0.8em; text-align:center; color:#666;">Speagle 2020, MNRAS 493, 3132 (arXiv:1904.02180), Fig. 2, top and bottom panels: one <code>dynesty</code> run. The live points shrink onto the posterior while $Z$ converges, with its error band.</div>

## Where the prior volume $X$ comes from

* $X(\lambda)$ is the prior mass where the likelihood beats $\lambda$: draw $\theta$ from the prior, compute $L(\theta)$, and $X(\lambda) = P(L > \lambda)$ - one minus the CDF of the likelihood values
* $L(X)$ is that function inverted: sort the prior draws by likelihood. $\int_0^1 L(X)\,dX$ is then just the average of $L$ over the prior - the definition of $P(D \mid M)$, now a 1-D integral whatever $d$ is

In [ ]:
# Where the prior volume X comes from, for the Occam toy with sigma_prior = 5 (a "skip" cell)
def plot_prior_volume(n=4000, seed=2):
    rng = np.random.default_rng(seed)
    th = rng.normal(0, 5.0, n)                             # draws from the prior
    lnL = np.sum(stats.norm.logpdf(occ_d[:, None], th, 1.0), axis=0)
    L = np.exp(lnL - lnL.max())
    lam = 0.3                                              # one likelihood threshold
    fig, (ax, az) = plt.subplots(1, 2, figsize=(12, 3.6))
    ax.scatter(th, L, s=4, c=np.where(L > lam, 'C3', '0.6'))
    ax.axhline(lam, color='C3', ls='--', lw=1.4)
    ax.text(-1.4, 0.7, f'$X$ = {np.mean(L > lam):.3f}:\nthe fraction of prior\ndraws above the line', color='C3', fontsize=12)
    ax.set_xlim(-1.5, 3); ax.set_xlabel('mean (draws from the prior)'); ax.set_ylabel('likelihood / max')
    Ls = np.sort(L)[::-1]; X = np.arange(1, n + 1) / n          # X(L) = P(L(theta) > L) under the prior
    az.plot(X, Ls, 'k', lw=2)
    az.fill_between(X, Ls, color='C0', alpha=0.3)
    az.plot(np.mean(L > lam), lam, 'o', color='C3', ms=8)
    az.set_xlim(0, 0.15); az.set_xlabel(r'$X$ = prior mass with likelihood above $L$ ($L \approx 0$ beyond 0.15)'); az.set_ylabel(r'$L(X)$ / max')
    az.set_title(r'shaded area = average $L$ over the prior = $P(D\mid M)$', fontsize=12)
    plt.tight_layout()
    plt.show()
    mm = np.linspace(-30, 30, 2001); lg = np.sum(stats.norm.logpdf(occ_d[:, None], mm, 1.0), axis=0)
    grid = np.log(np.trapezoid(np.exp(lg - lg.max()) * stats.norm.pdf(mm, 0, 5.0), mm)) + lg.max()
    print(f"average likelihood over {n} prior draws: ln P(D|M) = {np.log(np.mean(L)) + lnL.max():.2f}  (grid: {grid:.2f})")

In [ ]:
# left: prior draws and their likelihoods, red above one threshold; right: the same draws sorted, L against X
plot_prior_volume()

* the $X$ of a prior draw is uniform on $[0, 1]$ (the same trick as turning any CDF into a uniform). The worst of $n_{\rm live}$ uniform points sits at about $1 - 1/n_{\rm live}$ of the current $X$: that is the $e^{-1/n_{\rm live}}$ shrink
* nested sampling never computes $X$; it only needs new draws from the prior with $L > L_{\min}$. Codes draw a uniform $u$ in a unit cube and map it to $\theta$ through each prior's inverse CDF (`dynesty`'s `prior_transform`)
* so the prior must be written as such a map: easy for uniform, log-uniform or Gaussian priors, one parameter at a time; a correlated or empirical prior needs you to build the transform. The toy did this: a Gaussian's inverse CDF, restricted to $L > L_{\min}$

## Nested sampling, by hand

* $M_1$ with $\sigma_{\rm prior} = 5$, 100 live points, 900 kills, about 30 lines of numpy. Slide from 200 to 900 iterations

In [ ]:
# A bare-bones nested sampler for the same toy, M1 with sigma_prior = 5 (a "skip" cell)
def nested_sample(sigma_prior=5.0, n_live=100, n_iter=900, seed=0):   # 20 seeds average -29.17 +/- 0.13 against the grid's -29.18
    rng = np.random.default_rng(seed)
    lnL = lambda m: np.sum(stats.norm.logpdf(occ_d[:, None], m, 1.0), axis=0)
    live = rng.normal(0, sigma_prior, n_live)              # start: n_live draws from the prior
    live_lnL = lnL(live)
    dead, dead_lnL, lnX = [], [], []
    for k in range(n_iter):
        w = np.argmin(live_lnL)                            # the worst live point dies...
        dead.append(live[w]); dead_lnL.append(live_lnL[w]); lnX.append(-(k + 1) / n_live)   # ...and the prior volume shrinks by ~e^(-1/n_live)
        # replace it with a draw from the prior, restricted to where L beats the dead point. In 1-D that region is an
        # interval around the sample mean, so we can draw from it exactly; in many dimensions this step is the hard part
        r = np.sqrt(2 * (lnL(np.array([occ_mean]))[0] - live_lnL[w]) / occ_n)
        lo, hi = (occ_mean - r) / sigma_prior, (occ_mean + r) / sigma_prior
        live[w] = stats.truncnorm.rvs(lo, hi, scale=sigma_prior, random_state=rng)
        live_lnL[w] = lnL(live[w:w + 1])[0]
    lnX = np.array(lnX); dX = np.exp(np.concatenate([[0], lnX[:-1]])) - np.exp(lnX)
    lnZ_run = np.logaddexp.accumulate(np.array(dead_lnL) + np.log(dX))   # sum of L x (shell of prior volume), so far
    # H, the information: how many nats the data shrank the prior by. It sets nested sampling's error bar
    H = np.sum(np.exp(np.array(dead_lnL) + np.log(dX) - lnZ_run[-1]) * (np.array(dead_lnL) - lnZ_run[-1]))
    return np.array(dead), np.array(dead_lnL), lnX, lnZ_run, np.sqrt(H / n_live)   # Skilling's error bar on ln Z
ns = nested_sample()
# exact ln P(D | M1) for this toy, for comparison
_mm = np.linspace(-30, 30, 200001)
_l = np.sum(stats.norm.logpdf(occ_d[:, None], _mm[::100], 1.0), axis=0)
ns_lnZ_grid = np.log(np.trapezoid(np.exp(_l - _l.max()) * stats.norm.pdf(_mm[::100], 0, 5.0), _mm[::100])) + _l.max()
def plot_nested(iteration=300):
    dead, dl, lnX, lnZ, err = ns
    k = iteration
    fig, (ax, az, aw) = plt.subplots(1, 3, figsize=(13, 3.7))
    mm = np.linspace(-1, 2.5, 400)
    ll = np.sum(stats.norm.logpdf(occ_d[:, None], mm, 1.0), axis=0)
    ax.plot(mm, np.exp(ll - ll.max()), 'k', lw=1.5, label='likelihood')
    ax.scatter(dead[:k], np.exp(dl[:k] - ll.max()), c=np.arange(k), cmap='viridis', s=12, label='dead points')
    ax.axhline(np.exp(dl[k - 1] - ll.max()), color='C3', lw=1.4, ls='--', label=r'current $L_{\min}$')
    ax.set_xlabel('mean'); ax.set_ylabel('likelihood / max'); ax.set_xlim(-1, 2.5); ax.set_ylim(0, 1.45)
    ax.legend(frameon=False, fontsize=10, loc='upper left', ncol=3, handletextpad=0.3, columnspacing=0.8)
    LX = np.exp(dl + lnX - (dl + lnX).max())               # L x X: how much each step of ln X adds to the integral
    az.plot(-lnX, LX, color='0.75', lw=1)
    az.fill_between(-lnX[:k], LX[:k], color='C0', alpha=0.35); az.plot(-lnX[:k], LX[:k], 'C0', lw=2)
    az.set_xlim(0, -lnX[-1]); az.set_xlabel(r'$-\ln X$, X = prior volume with $L > L_{\min}$')
    az.set_ylabel(r'$L \times X$ (scaled)'); az.set_title('shaded area = $P(D\\mid M_1)$ so far', fontsize=12)
    aw.plot(np.arange(1, len(lnZ) + 1), lnZ, color='0.6', lw=1)
    aw.plot(np.arange(1, k + 1), lnZ[:k], 'C3', lw=2.2)
    aw.axhline(ns_lnZ_grid, color='k', ls=':', lw=1.6, label=f'grid: {ns_lnZ_grid:.2f}')
    aw.axhspan(lnZ[-1] - err, lnZ[-1] + err, color='C3', alpha=0.12, label=f'nested: {lnZ[-1]:.2f} $\\pm$ {err:.2f}')
    aw.set_ylim(ns_lnZ_grid - 12, ns_lnZ_grid + 2); aw.set_xlabel('iteration'); aw.set_ylabel(r'$\ln P(D \mid M_1)$ so far')
    aw.legend(frameon=False, loc='lower right', fontsize=11); aw.set_title(f'iteration {k}: {lnZ[k-1]:.2f}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# left: each dead point sits higher on the likelihood; middle: L against the prior volume X; right: ln P(D|M1) converging to the grid answer
controls_right(plot_nested, iteration=IntSlider(value=300, min=200, max=900, step=20, continuous_update=False, description='iteration'))

* each new dead point sits higher on the likelihood, from the prior's tails up to the peak
* middle: $L \times X$ against $-\ln X$ has a bump, because early on $L$ is tiny and late on $X$ is tiny. Since $X\,d(\ln X) = dX$, the area under it is the evidence
* the answer, $-29.18 \pm 0.16$, agrees with the grid's $-29.18$ (20 runs with other seeds scatter by $\pm 0.13$). The error bar is $\sqrt{H/n_{\rm live}}$, where $H$ is how much the data shrank the prior (in nats, natural-log units). This is why `dynesty` reports an uncertainty on $\ln Z$

## 47 UMa, one planet at a time

* 47 UMa is a Sun-like star whose radial velocity wobbles as planets tug on it - the same measurement as Lab 05
* Gregory &amp; Fischer (2010): 220 Lick Observatory velocities over 21.6 years (their Tables 1-2), here with the planets from their 3-planet fit (Table 4) taken out one at a time
* a periodogram fits a sinusoid at each trial period (Day 9: "this fit, one frequency at a time") and plots how much of the scatter it explains. Peaks are candidate periods. Lomb-Scargle is the version for unevenly spaced data like these
* right: the periodogram of the residuals. Red dotted lines are planets not yet subtracted. Step the slider from 0 to 3 planets

In [ ]:
# The real Lick velocities, minus 0-3 planets from Gregory & Fischer's 3-planet fit (Table 4, MAP values) (a "skip" cell)
from astropy.timeseries import LombScargle
_f = 'data/gregory2010_47uma_lick_rv.csv'
with open(_f) as _fh:
    _nhead = sum(1 for _line in _fh if _line.startswith('#'))
_u = np.genfromtxt(_f, delimiter=',', skip_header=_nhead, names=True)
uma_t, uma_v, uma_e = _u['jd_minus_2440000'], _u['rv_mps'], _u['rv_err_mps']
# P [d], K [m/s], e, omega [deg], periastron [JD - 2440000]; added in the order a periodogram finds them
UMA_PLANETS = [dict(P=1079.2, K=50.3, e=0.012, w=345, tp=11943), dict(P=21342, K=13.2, e=0.44, w=111, tp=12047),
               dict(P=2278, K=9.6, e=0.48, w=222, tp=11930)]
def kepler_rv(t, P, K, e, w, tp):
    M = 2 * np.pi * (t - tp) / P
    E = M.copy()
    for _ in range(50):                                    # Newton's method for Kepler's equation, M = E - e sin E
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    nu = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    return K * (np.cos(nu + np.radians(w)) + e * np.cos(np.radians(w)))
def plot_uma_rv(planets=0):
    model = lambda t: sum((kepler_rv(t, **p) for p in UMA_PLANETS[:planets]), np.zeros_like(t))
    r = uma_v - model(uma_t); off = np.average(r, weights=uma_e**-2); r -= off
    tt = np.linspace(uma_t.min(), uma_t.max(), 3000)
    fig, (ax, az, aw) = plt.subplots(1, 3, figsize=(13, 3.7), gridspec_kw=dict(width_ratios=[1.3, 1.3, 1]))
    ax.errorbar(uma_t / 365.25 + 1968.0, uma_v, yerr=uma_e, fmt='o', color='0.4', ms=3, elinewidth=0.6)
    ax.plot(tt / 365.25 + 1968.0, model(tt) + off, 'C3', lw=1.6)
    ax.set_xlabel('year'); ax.set_ylabel('RV [m/s]'); ax.set_title(f'data and the {planets}-planet model', fontsize=12)
    az.errorbar(uma_t / 365.25 + 1968.0, r, yerr=uma_e, fmt='o', color='C0', ms=3, elinewidth=0.6)
    az.axhline(0, color='k', lw=0.8); az.set_ylim(-110, 110)
    az.set_xlabel('year'); az.set_ylabel('residual [m/s]'); az.set_title(f'{planets} subtracted: rms {np.sqrt(np.mean(r**2)):.1f} m/s', fontsize=12)
    per = np.logspace(np.log10(100), np.log10(30000), 20000)
    pw = LombScargle(uma_t, r, uma_e).power(1 / per)
    aw.axvspan(355, 380, color='C2', alpha=0.25, lw=0, zorder=0, label='~1 year')
    aw.semilogx(per, pw, 'k', lw=0.9)
    for p in UMA_PLANETS[planets:]:
        aw.axvline(p['P'], color='C3', ls=':', lw=1.6)
    aw.set_ylim(0, 1.25 * max(pw.max(), 0.1)); aw.set_xlabel('period [d]'); aw.set_ylabel('periodogram power')
    aw.legend(frameon=False, loc='center right', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# left: Lick velocities and the model; middle: residuals; right: their periodogram (red dotted: planets not yet subtracted)
controls_right(plot_uma_rv, planets=IntSlider(value=0, min=0, max=3, step=1, continuous_update=False, description='planets'))

* three planets take the rms from 35 to 6.7 m/s (they quote 6.5 from their full fit; their table rounds the orbits)
* the residuals peak near 1 yr. Gregory &amp; Fischer's 4th signal is at 371 d, suspiciously close to a year - the period of Earth's motion and of the observing seasons, a classic place for an artifact of the data reduction
* so is the 4th signal a planet? Gregory &amp; Fischer computed $P(D \mid M)$ for 0 to 4 planets

## 47 UMa's Bayes factors

* each dot is one posterior sample of a planet's period and eccentricity; step the slider through 2, 3 and 4 planets

In [ ]:
# Gregory & Fischer's posterior samples for the 2-, 3- and 4-planet models, one figure per model (a "skip" cell)
from IPython.display import Image as _Image
UMA_FIGS = {2: ('images/gregory2010_fig8_47uma_2planet_ecc_vs_period.png', 'Fig. 8: 2 planets. The reference model, B = 1'),
            3: ('images/gregory2010_fig11_47uma_3planet_ecc_vs_period.png', r'Fig. 11: 3 planets. B(3 vs 2) $\gtrsim 10^{17}$'),
            4: ('images/gregory2010_fig13_47uma_4planet_ecc_vs_period.png', 'Fig. 13: 4 planets, a new one at 371 d. B(4 vs 3) ≈ 1')}
def show_uma_model(planets=3):
    f, txt = UMA_FIGS[planets]
    print(txt.replace(r'$\gtrsim 10^{17}$', '≳ 10^17'))
    display(_Image(filename=f, width=720))

In [ ]:
# eccentricity against period for each posterior sample, one figure per model; the printed line is that model's Bayes factor
controls_right(show_uma_model, planets=IntSlider(value=2, min=2, max=4, step=1, continuous_update=False, description='planets'))

<div style="font-size:0.8em; text-align:center; color:#666;">Gregory &amp; Fischer 2010, MNRAS 403, 731 (arXiv:1003.5549), Figs. 8, 11 and 13; Bayes factors from their Table 5 and §5.</div>

* two planets to three: $B \gtrsim 10^{17}$. Lick changed detectors several times, and each setup can have its own velocity zero point; fit those four extra offsets and $B$ drops to $2\times10^5$ - still very strong
* a 4th planet at 371 d drops the rms from 6.5 to 6.1 m/s, yet $B \approx 1$
* the 4th signal vanished when McDonald Observatory data were added, so declining to decide was the right call

**IMPORTANT: a lower rms by itself doesn't tell you whether the planet is real!**

## Model comparison in cosmology

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<div style="flex:0 0 38%;">

<img src="images/trotta2008_fig3_bayes_factor_plane.png" alt="Trotta's plane for nested model comparison: horizontal axis is the information content of the data, log10 of prior width over posterior width; vertical axis is how many sigma the extra parameter is from its null value; contours of the log Bayes factor divide the plane into regions favoring the simpler model (wide prior, weak detection) and the more complex model (strong detection)" style="max-height:340px">

<div style="font-size:0.75em; color:#666;">Trotta 2008, Contemp. Phys. 49, 71 (arXiv:0803.4089), Fig. 3: the outcome depends on how much the data shrank the prior (horizontal) and how many $\sigma$ the detection is (vertical).</div>

</div>
<div>

* the early universe's density ripples (seen in the CMB, the microwave background) have the same strength on every scale if $n_s = 1$. $M_0$: $n_s = 1$ exactly. $M_1$: $n_s$ free, with a prior of $0.8$ to $1.2$ from inflation theory
* WMAP (a CMB satellite), first year: $2\ln B_{1,0}$ from $-0.8$ to $+2.2$, weak. Three years plus galaxy surveys: $+4.0$ to $+5.8$, positive for a tilt
* Trotta quotes $\ln B_{01}$, model 0 over model 1; we flipped the sign and doubled it for our ladder
* his Eq. 26 is our $\sigma_{\rm prior}$ formula: $2\ln B_{1,0} \approx \lambda^2 - 2\ln(\sigma_{\rm prior}/\sigma_{\rm post})$
* Planck (the next CMB satellite): $n_s = 0.965 \pm 0.004$, so $\lambda \approx 9$; with $\sigma_{\rm prior}/\sigma_{\rm post} \approx 0.2/0.004 = 50$, $2\ln B_{1,0} \approx 77 - 8 \approx +70$, off the top of the ladder

</div>
</div>

## Aside - a Bayes factor from four data points

<img src="images/poon2025_fig6_obliquity_bayes_factor.png" alt="Two panels: left, the posterior on the concentration parameter kappa for four young super-Jupiters, piled up near zero (spins pointing anywhere), with the prior on kappa dashed; right, bars showing the Bayes factor favoring isotropic spins for each subset of the four systems, rising to 15 for all four together" style="display:block;margin:0 auto;max-height:230px">

<div style="font-size:0.8em; text-align:center; color:#666;">Poon, Bryan, Rein, Dong, Speagle &amp; Pham 2025, ApJL (arXiv:2511.04091), Fig. 6.</div>

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<div>

* do young super-Jupiters spin in line with their orbits, like planets, or point anywhere, like brown dwarfs? $\kappa$ sets how tightly spins cluster on the orbit's axis ($\kappa = 5$ planet-like, $\kappa = 0$ random)
* four systems give $B = 15$ for "anywhere" ($2\ln B = 5.4$, positive). Independent data multiply, so 15 is the product of the four, $4.0 \times 2.9 \times 1.3 \times 0.97$
* swap $\kappa = 5$ for 4 or 6 and $B$ becomes 8 or 24 - so they report how the answer moves

</div>
<div style="flex:0 0 22%; text-align:center;">

<img src="images/faculty_jiayin_dong.jpg" alt="Portrait of Jiayin Dong" style="max-height:200px">

<div style="font-size:0.75em; color:#666;">Co-author Jiayin Dong, Assistant Professor of Astronomy here at Illinois. Photo: Illinois Astronomy.</div>

</div>
</div>

## Today, 3:45 pm: Dominic Benford (NASA)

<div style="display:flex; align-items:flex-start; gap:1.4em; margin-top:0.4em;">
<div>

* **"As the Nancy Grace Roman Space Telescope Graduates from Construction to Discoveries"**
* Roman's Galactic Bulge Time Domain Survey will find planets by microlensing: each candidate is a model comparison
* 3:45 pm, here in 134 Astronomy; refreshments from 3:30 pm in Astronomy 222

</div>
<div style="flex:0 0 36%;">

<img src="images/roman_telescope.jpg" alt="Rendering of the Nancy Grace Roman Space Telescope spacecraft, a cylindrical telescope barrel with solar panels" style="max-height:260px">

<div style="font-size:0.75em; color:#666;">NASA's Goddard Space Flight Center.</div>

</div>
</div>

<div style="font-size:0.7em; color:#666;">Title: calendars.illinois.edu, event 33563000 (checked 2026-10-03). Listing: astro.illinois.edu/news-events/astrophysics-colloquium.</div>

## Wrapping up

You should be comfortable with:

* the review test: WLS's $\chi^2/\nu = 5$ was the warning
* checking a chain's trace, acceptance and $\tau$ before quoting its percentiles
* $P(D \mid M)$, the evidence, and the Bayes factor as a ratio of two
* $2\ln B$ on Day 8's ladder, and the Occam factor that sets it. An extra parameter is worth it only if the fit improves by more than its prior costs
* grid, Laplace (BIC is its cheapest form), nested sampling
* today's question: $\sigma_{\rm int}$ was worth it ($\chi^2/\nu = 5$ demanded it); Lab 05's jitter you test yourself; the 4th planet around 47 UMa has $B \approx 1$, so not yet; Planck's $2\ln B \approx +70$ says the tilt in $n_s$ is real

Thu Oct 8: an in-class exercise (a line or a quadratic?), then detailed balance, $\hat R$ across several chains, and HMC with `PyMC`. Lab 05 due Fri Oct 9

## Lab defenses

* two of you each class, both up front together, here in the room, in the last ~10 minutes
* Abha and I ask both of you about your most recently graded lab: your choices and your checks
* everyone else listens - these are the questions you will get too
* today: **Zahi Rahman and Grace Houlihan**
* Thursday: **Anuva Ghosalkar and KC Ciaccio** (moved from today, because three students dropped the class)